### Connect to Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

# If using Drive:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/smart-assessor-fdeaa-firebase-adminsdk-fbsvc-cd76bf92b6.json"

os.environ["FIREBASE_PROJECT_ID"] = "smart-assessor-fdeaa"



Mounted at /content/drive


### Set paths to model and adapter

In [ ]:
import os, pathlib, shutil

MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1"   # base model goes here
ADAPTER_DIR = "/content/drive/MyDrive/SmartAssessor/wizardmath-lora-checkpoints"  # LoRA adapters

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("MODEL_DIR =", MODEL_DIR)
print("ADAPTER_DIR =", ADAPTER_DIR)




MODEL_DIR = /content/drive/MyDrive/models/WizardMath-7B-V1.1
ADAPTER_DIR = /content/drive/MyDrive/SmartAssessor/wizardmath-lora-checkpoints


### Libraries installation

In [ ]:
!pip -q install transformers peft accelerate bitsandbytes langchain langchain_community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### Load base model from Drive and attach LoRA adapter

In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32
device_map = "auto" if device == "cuda" else None

# Tokenizer from Drive (offline)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 8-bit quantization on GPU; CPU loads standard weights
bnb_config = BitsAndBytesConfig(load_in_8bit=True, llm_int8_enable_fp32_cpu_offload=True) if device=="cuda" else None

# Base model from Drive (offline)
if bnb_config is not None:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch_dtype,
        trust_remote_code=True
    )
else:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=torch_dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )

# Attach LoRA adapters from Drive
model = PeftModel.from_pretrained(base, ADAPTER_DIR, torch_dtype=torch_dtype)
if device_map is None:
    model = model.to(device)
model.eval()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.5 MB/s eta 0:00:00


### Merge adapter and base model

In [ ]:
# Your existing code to load the base model and attach adapters...
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# (Assuming all your variables like MODEL_DIR, ADAPTER_DIR, model, and tokenizer are already loaded)

print("Model loaded with adapters. Now merging...")

# 1. Merge the LoRA weights into the base model
# This returns a new model with the merged weights.
merged_model = model.merge_and_unload()

print("Merging complete.")

# 2. Define a path to save your new, self-contained model
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

# 3. Save the merged model and its tokenizer
print(f"Saving merged model to {MERGED_MODEL_DIR}...")
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print("Merged model saved successfully!")

KeyboardInterrupt: 

### build the agent - a variation we tried with langchain as a framework and set of tools to define our agent

In [ ]:
# # -*- coding: utf-8 -*-
# """
# LangChain Linear Algebra Assessor using a self-contained, merged Hugging Face model.

# This script loads a pre-merged model (base + LoRA adapter) and uses it within a
# LangChain agent to solve and verify linear algebra problems.
# """

# # --- Prerequisite Installations ---
# # !pip -q install langchain langchain-community sympy transformers accelerate torch

# import re
# import math
# import json
# import torch
# from typing import List, Dict, Any

# # --- 1) Load the self-contained, merged model ---
# # This section is the primary modification.
# # We no longer assume a model is in memory; we load it directly.

# from transformers import AutoTokenizer, AutoModelForCausalLM

# # Define the path to your saved merged model
# MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

# print("Loading tokenizer and merged model from Google Drive...")
# try:
#     tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)

#     # Load the model in half-precision and map it automatically to the GPU
#     model = AutoModelForCausalLM.from_pretrained(
#         MERGED_MODEL_DIR,
#         torch_dtype=torch.float16,
#         device_map="auto"
#     )
#     # Set the model to evaluation mode (important for inference)
#     model.eval()
#     print("Merged model loaded successfully! ✅")

# except Exception as e:
#     print(f"Error loading model: {e}")
#     print(f"Please ensure the path '{MERGED_MODEL_DIR}' is correct and you have mounted Google Drive.")
#     # Exit if model fails to load
#     exit()


# # --- 2) Wrap the HF model in a LangChain LLM ---
# from transformers import pipeline
# from langchain_community.llms import HuggingFacePipeline

# # The HuggingFacePipeline will now use our loaded merged model
# device_idx = 0 if torch.cuda.is_available() else -1
# gen_pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=256,
#     do_sample=False,
#     pad_token_id=tokenizer.eos_token_id,
#     eos_token_id=tokenizer.eos_token_id,
# )
# llm = HuggingFacePipeline(pipeline=gen_pipe)
# print("LangChain pipeline created.")


# # --- 3) A safe SymPy helper for the agent ---
# from sympy import symbols, Matrix, Rational, sqrt, det, nsimplify, N
# from sympy.parsing.sympy_parser import parse_expr

# def _safe_eval_sympy(expr: str) -> str:
#     """
#     Evaluate a SymPy expression for linear algebra / algebraic numerics.
#     Allowed names include Matrix, Rational, sqrt, det, etc.
#     Returns both exact and float forms when possible.
#     """
#     allowed = {
#         "Matrix": Matrix, "Rational": Rational, "sqrt": sqrt, "det": det,
#         "N": N, "nsimplify": nsimplify,
#         "x": symbols("x"), "y": symbols("y"), "z": symbols("z"),
#         "a": symbols("a"), "b": symbols("b"), "c": symbols("c"),
#     }
#     try:
#         expr_obj = parse_expr(expr, local_dict=allowed, evaluate=True)
#         exact = nsimplify(expr_obj)
#         try:
#             approx = float(N(exact))
#             return json.dumps({"exact": str(exact), "approx": approx})
#         except Exception:
#             return json.dumps({"exact": str(exact)})
#     except Exception as e:
#         return json.dumps({"error": f"SymPy error: {e}"})


# # --- 4) Register a LangChain Tool for the agent ---
# from langchain.tools import tool

# @tool("sympy_solve", return_direct=False)
# def sympy_solve(expr: str) -> str:
#     """Compute a linear-algebra/algebra expression using SymPy.
#     Provide a single Python/SymPy expression, e.g.:
#     - 'det(Matrix([[2, -1],[3,4]]))'
#     - 'Matrix([[1,2],[3,4]]).eigenvals()'
#     - 'N(det(Matrix(A)))' after you define A in the expression."""
#     return _safe_eval_sympy(expr)

# tools = [sympy_solve]


# # --- 5) Build an agent that can call the tool ---
# from langchain.agents import AgentType, initialize_agent

# system_rules = """You are a careful linear algebra solver.
# - Use the tool sympy_solve for exact numeric steps when helpful.
# - Show brief steps. At the end ALWAYS output:
# Final Answer: <number>
# Short Solution: <one concise line>"""

# agent = initialize_agent(
#     tools=tools,
#     llm=llm,
#     agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
#     verbose=False,
#     handle_parsing_errors=True,
#     agent_kwargs={"system_message": system_rules}
# )
# print("LangChain agent initialized.")


# # --- 6) Helper to extract the final answer from the text ---
# _num_re = re.compile(r"Final Answer:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)")

# def extract_final_and_short(text: str):
#     final = None
#     m = _num_re.search(text)
#     if m:
#         try:
#             final = float(m.group(1))
#         except (ValueError, TypeError):
#             final = None
#     # Grab short solution (optional)
#     short = ""
#     ms = re.search(r"Short Solution:\s*(.*)", text)
#     if ms:
#         short = ms.group(1).strip()
#     return final, short


# # --- 7) Main assessment function ---
# def assess(problems: List[Dict[str, Any]], rel_tol: float = 1e-6, abs_tol: float = 1e-6):
#     """
#     problems: list of {"question": str, "given_answer": float/int}
#     Prints per-problem result.
#     """
#     for i, item in enumerate(problems, 1):
#         q = item["question"]
#         given = float(item["given_answer"])
#         prompt = f"""Problem: {q}

# Requirements:
# - If helpful, call sympy_solve with a single SymPy expression to compute determinants, eigenvalues, ranks, solutions to systems, etc.
# - Keep steps short.
# - End with:
# Final Answer: <number>
# Short Solution: <one concise line>
# """
#         try:
#             result = agent.invoke({"input": prompt})
#             text = result["output"] if isinstance(result, dict) and "output" in result else str(result)
#         except Exception as e:
#             text = f"ERROR: {e}"

#         final_num, short_sol = extract_final_and_short(text)

#         print(f"\n--- Problem {i} ---")
#         print(f"Question: {q}")
#         print(f"Given numeric answer: {given}")

#         if final_num is None:
#             print("❌ Could not parse a numeric 'Final Answer'. The agent output was:")
#             print(text)
#             continue

#         correct = math.isclose(final_num, given, rel_tol=rel_tol, abs_tol=abs_tol)
#         if correct:
#             print(f"✅ Correct answer: {final_num}")
#         else:
#             print(f"❌ Incorrect. Correct answer is: {final_num}")
#             if short_sol:
#                 print(f"Short solution: {short_sol}")
#             else:
#                 tail = "\n".join(text.strip().splitlines()[-4:])
#                 print("Short solution:", tail)


# # --- 8) Example usage ---
# sample_problems = [
#     {
#         "question": "Compute det([[ -3, 2 ], [ 1, 4 ]]).",
#         "given_answer": -14
#     },
#     {
#         "question": "Find the determinant of [[2, -1], [3, 4]].",
#         "given_answer": 11
#     },
#     {
#         "question": "What is the determinant of the 2x2 matrix with rows [5, 1] and [2, 3]?",
#         "given_answer": 13
#     }
# ]

# # Run the assessment
# if __name__ == '__main__':
#     # Make sure you have mounted Google Drive in your Colab notebook
#     # from google.colab import drive
#     # drive.mount('/content/drive')
#     assess(sample_problems)


Loading tokenizer and merged model from Google Drive...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merged model loaded successfully! ✅


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/tmp/ipython-input-3437813674.py:63: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=gen_pipe)


...

Token indices sequence length is longer than the specified maximum sequence length for this model (2054 > 2020). Running this sequence through the model will result in indexing errors



--- Problem 1 ---
Question: Compute det([[ -3, 2 ], [ 1, 4 ]]).
Given numeric answer: -14.0
❌ Could not parse a numeric 'Final Answer'. The agent output was:
ERROR: CUDA out of memory. Tried to allocate 2.25 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.63 GiB is free. Process 60520 has 13.11 GiB memory in use. Of the allocated memory 11.86 GiB is allocated by PyTorch, and 1.12 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



--- Problem 2 ---
Question: Find the determinant of [[2, -1], [3, 4]].
Given numeric answer: 11.0
❌ Could not parse a numeric 'Final Answer'. The agent output was:
ERROR: CUDA out of memory. Tried to allocate 1.91 GiB. GPU 0 has a total capacity of 14.74 GiB of which 228.12 MiB is free. Process 60520 has 14.52 GiB memory in use. Of the allocated memory 11.95 GiB is allocated by PyTorch, and 2.43 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


KeyboardInterrupt: 

### version 1

In [ ]:
"""
Two-LLM Micro-Agent (WizardMath solver + tiny parser)
- WizardMath (GPU, fp16) SOLVES the math and writes the final two lines:
    Final Answer: <number>
    Short Solution: <one line>
- Tiny parser LLM (CPU) normalizes that text into JSON {final, short}.
- If the parser LLM hiccups, a robust regex fallback parses the answer.

Notes:
- No LangChain, no tools, no SymPy verification (WizardMath alone is the solver).
- Includes all stability patches: fp16 dtype, logging silence, few-shot footer forcing,
  balanced-JSON extractor, and broader regex.
"""

# =========================
# Optional installs for Colab (uncomment if needed)
# =========================
# !pip -q install transformers accelerate torch

import re
import math
import json
import torch
from typing import List, Dict, Any
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

# Silence HF warnings
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# =========================
# 1) Load WizardMath (solver) on GPU, fp16
# =========================
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

print("Loading WizardMath tokenizer + model from Google Drive...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_DIR,
        dtype=torch.float16,       # <- use `dtype`, not deprecated `torch_dtype`
        device_map="auto"
    )
    model.eval()
    print("WizardMath loaded ✅")
except Exception as e:
    print(f"Error loading WizardMath: {e}")
    raise

# =========================
# 2) Load tiny parser LLM on CPU (very lightweight)
# =========================
PARSER_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # alt: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading parser model on CPU: {PARSER_MODEL_ID} ...")
try:
    parser_tok = AutoTokenizer.from_pretrained(PARSER_MODEL_ID, trust_remote_code=True)
    parser_model = AutoModelForCausalLM.from_pretrained(
        PARSER_MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )
    parser_pipe = pipeline(
        "text-generation",
        model=parser_model,
        tokenizer=parser_tok,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=parser_tok.eos_token_id,
        eos_token_id=parser_tok.eos_token_id,
    )
    print("ParserAgent loaded on CPU ✅")
except Exception as e:
    print(f"⚠️ Parser LLM failed to load: {e}")
    parser_pipe = None  # Will fall back to regex-only

# =========================
# 3) Build solver + parser pipelines
# =========================

# 3a) Solver (WizardMath) — keep outputs short/deterministic
solver_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=192,       # short to avoid context blowups
    do_sample=False,          # greedy is more consistent for math
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

SOLVER_SYS = (
    "You are WizardMath. Solve the question clearly.\n"
    "End your message with two lines exactly:\n"
    "Final Answer: <number>\n"
    "Short Solution: <one line>"
)

# single ultra-short example to force footer format
FEWSHOT = (
    "Example\n"
    "Question: Compute the determinant of [[1,2],[3,4]]\n"
    "Final Answer: -2\n"
    "Short Solution: det = 1*4 - 2*3"
)

def solve_with_wizard(question: str) -> str:
    prompt = f"{SOLVER_SYS}\n\n{FEWSHOT}\n\nQuestion: {question}\n"
    out = solver_pipe(prompt, return_full_text=False)[0]["generated_text"]
    return out.strip()

# 3b) Parser (tiny LLM → JSON) + robust fallback regex
PARSER_SYS = (
    'Extract the final numeric answer and the short solution from the text. '
    'Return ONLY one JSON line exactly like: '
    '{"final": <number>, "short": "<one line>"} '
    '(If missing, set final to null and short to ""). No extra text.'
)

def extract_first_json_balanced(s: str) -> str:
    """Extract the first balanced {...} JSON object from s."""
    start = s.find("{")
    if start == -1:
        raise ValueError("no JSON object found")
    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return s[start:i+1]
    raise ValueError("unbalanced JSON")

# broader patterns (covers "Final Answer:", "The answer is:", "Answer:")
_answer_pat = re.compile(
    r"(?:Final\s*Answer\s*:|The\s*answer\s*is\s*:|Answer\s*:)\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
    re.IGNORECASE,
)
_short_pat = re.compile(
    r"(?:Short\s*Solution\s*:|Explanation\s*:)\s*(.*)",
    re.IGNORECASE,
)

def parse_with_llm_or_regex(raw: str) -> Dict[str, Any]:
    # Preferred: parser LLM on CPU -> strict JSON
    if parser_pipe is not None:
        prompt = f"{PARSER_SYS}\n\n===BEGIN===\n{raw}\n===END===\nJSON:"
        try:
            jtxt = parser_pipe(prompt, return_full_text=False)[0]["generated_text"].strip()
            obj = json.loads(extract_first_json_balanced(jtxt))
            final = obj.get("final", None)
            if isinstance(final, str):
                try:
                    final = float(final)
                except Exception:
                    final = None
            short = (obj.get("short") or "").strip()

            # Soft backfill from regex if any field missing
            if final is None or not short:
                fm = _answer_pat.search(raw)
                sm = _short_pat.search(raw)
                if final is None and fm:
                    try:
                        final = float(fm.group(1))
                    except Exception:
                        pass
                if not short and sm:
                    short = sm.group(1).strip()
            return {"final": final, "short": short}
        except Exception:
            # fall through to regex
            pass

    # Fallback: robust regex parsing
    fm = _answer_pat.search(raw)
    sm = _short_pat.search(raw)
    final = float(fm.group(1)) if fm else None
    short = sm.group(1).strip() if sm else ""
    return {"final": final, "short": short}

print("Two-agent micro-pipeline ready (Solver=WizardMath on GPU, Parser=tiny model on CPU).")

# =========================
# 4) Assessment (one-by-one)
# =========================
def assess(problems: List[Dict[str, Any]], rel_tol: float = 1e-6, abs_tol: float = 1e-6):
    for i, item in enumerate(problems, 1):
        q = item["question"]
        given = float(item["given_answer"])

        print(f"\n--- Problem {i} ---")
        print(f"Question: {q}")
        print(f"Given numeric answer: {given}")

        try:
            raw = solve_with_wizard(q)             # Agent 1: WizardMath solves
            parsed = parse_with_llm_or_regex(raw)  # Agent 2: parser normalizes
            final_num = parsed["final"]
            short_sol = parsed["short"]
        except Exception as e:
            print(f"❌ Pipeline failure: {e}")
            continue

        if final_num is None:
            print("❌ Could not parse a numeric 'Final Answer'. Full solver output:")
            print(raw)
            continue

        if math.isclose(final_num, given, rel_tol=rel_tol, abs_tol=abs_tol):
            print(f"✅ Correct answer: {final_num}")
        else:
            print(f"❌ Incorrect. Solver's answer is: {final_num}")
            if short_sol:
                print(f"Short solution: {short_sol}")
            # If you want to see the full text from the solver:
            # print("---- Solver full output ----")
            # print(raw)

# =========================
# 5) Example usage
# =========================
sample_problems = [
    {"question": "Compute the determinant of the matrix [[8, 10], [-11, -5]]", "given_answer": 70},
    {"question": "Compute the determinant of the matrix [[-2, 1], [7, -1]]", "given_answer": -5},
    {"question": "Calculate the determinant of the matrix [[4, 6, 2], [6, -6, -5], [3, 3, -1]]", "given_answer": 102},
    {"question": "Calculate the determinant of the matrix [[0, 2, 6], [-4, -4, -5], [3, 0, 1]]", "given_answer": 60},
    {"question": "Calculate the determinant of the matrix [[-7, -4, 6], [0, -4, -7], [-7, 1, 4]]", "given_answer": -301},
]

if __name__ == "__main__":
    # from google.colab import drive
    # drive.mount('/content/drive')
    assess(sample_problems)


CUDA available: True
Using GPU: Tesla T4
Loading WizardMath tokenizer + model from Google Drive...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

WizardMath loaded ✅
🔧 IMPROVED SINGLE-LLM SOLVER LOADED
✅ Manual verification functions loaded
✅ Assessment function loaded
📋 Test problems loaded:
1. Compute the determinant of the matrix [[8, 10], [-11, -5]] (Expected: 70)
2. Compute the determinant of the matrix [[-2, 1], [7, -1]] (Expected: -5)
3. Calculate the determinant of the matrix [[4, 6, 2], [6, -6, -5], [3, 3, -1]] (Expected: 102)
4. Calculate the determinant of the matrix [[0, 2, 6], [-4, -4, -5], [3, 0, 1]] (Expected: 50)
5. Calculate the determinant of the matrix [[-7, -4, 6], [0, -4, -7], [-7, 1, 4]] (Expected: -301)
🎯 RUNNING IMPROVED MATH SOLVER ASSESSMENT

Key improvements over original:
- ❌ Removed problematic parser LLM
- 🔍 Enhanced debugging output
- ✅ Corrected test case expected answers
- 🎯 Better prompting strategy
- 🔬 Manual verification of all answers
🚀 STARTING ENHANCED MATH SOLVER ASSESSMENT

📝 PROBLEM 1
Question: Compute the determinant of the matrix [[8, 10], [-11, -5]]
Given expected answer: 70.0
✅ Manua

### version 2

In [ ]:
"""
IMPROVED Single-LLM Math Solver with Enhanced Debugging
- Uses only WizardMath (no problematic parser LLM)
- Regex-only parsing for reliability
- Enhanced debugging and logging
- Better prompting following HuggingFace guidelines
- Manual verification of test cases

Key improvements:
1. Removed parser LLM that was generating wrong solutions
2. Added comprehensive debugging output
3. Fixed test case with wrong expected answer (Problem 4: should be 50, not 60)
4. Better prompting strategy for determinant calculations
5. Clear visibility into solver reasoning
"""

import re
import math
import torch
from typing import List, Dict, Any, Optional
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

# Silence HF warnings
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# =========================
# 1) Load WizardMath (single solver) on GPU, fp16
# =========================
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

print("Loading WizardMath tokenizer + model from Google Drive...")
try:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_DIR,
        dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    print("WizardMath loaded ✅")
except Exception as e:
    print(f"Error loading WizardMath: {e}")
    raise

# =========================
# 2) Enhanced Solver Pipeline with Better Prompting
# =========================
solver_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,       # Allow more tokens for step-by-step reasoning
    do_sample=False,          # Deterministic for math
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# Improved system prompt following WizardMath best practices
SOLVER_SYS = (
    "You are WizardMath, an expert in linear algebra and matrix operations.\n"
    "Solve determinant problems step by step with clear calculations.\n"
    "Always end with: Final Answer: <number>"
)

def solve_with_wizard(question: str, debug: bool = True) -> Dict[str, Any]:
    """
    Solve a math problem with WizardMath and return detailed results.
    """
    # Create a clear, direct prompt
    prompt = f"{SOLVER_SYS}\n\nProblem: {question}\n\nSolution:"

    if debug:
        print(f"\n🔍 DEBUG - Prompt sent to solver:")
        print(f"{'='*50}")
        print(prompt)
        print(f"{'='*50}")

    try:
        # Generate solution
        output = solver_pipe(prompt, return_full_text=False)[0]["generated_text"]
        raw_output = output.strip()

        if debug:
            print(f"\n🔍 DEBUG - Raw solver output:")
            print(f"{'='*50}")
            print(raw_output)
            print(f"{'='*50}")

        return {
            "prompt": prompt,
            "raw_output": raw_output,
            "success": True,
            "error": None
        }
    except Exception as e:
        return {
            "prompt": prompt,
            "raw_output": "",
            "success": False,
            "error": str(e)
        }

# =========================
# 3) Regex-Only Parser (No LLM interference)
# =========================
def parse_answer_regex_only(text: str, debug: bool = True) -> Dict[str, Any]:
    """
    Parse the final answer using only regex patterns.
    """
    # Multiple patterns to catch different answer formats
    patterns = [
        r"Final\s*Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"The\s*answer\s*is\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"=\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)(?:\s|$)",
    ]

    final_answer = None
    matched_pattern = None

    for i, pattern in enumerate(patterns):
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            try:
                final_answer = float(match.group(1))
                matched_pattern = f"Pattern {i+1}: {pattern}"
                break
            except (ValueError, TypeError):
                continue

    if debug:
        print(f"\n🔍 DEBUG - Parsing results:")
        print(f"Final answer found: {final_answer}")
        print(f"Matched pattern: {matched_pattern}")

    return {
        "final_answer": final_answer,
        "matched_pattern": matched_pattern,
        "parsing_success": final_answer is not None
    }

print("🔧 IMPROVED SINGLE-LLM SOLVER LOADED")

# =========================
# Manual Verification Functions
# =========================
def manual_determinant_2x2(matrix: List[List[float]]) -> float:
    """Calculate 2x2 determinant: ad - bc"""
    return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]

def manual_determinant_3x3(matrix: List[List[float]]) -> float:
    """Calculate 3x3 determinant using cofactor expansion along first row"""
    a, b, c = matrix[0]
    d, e, f = matrix[1]
    g, h, i = matrix[2]

    return (a * (e * i - f * h) -
            b * (d * i - f * g) +
            c * (d * h - e * g))

def extract_matrix_from_question(question: str) -> List[List[float]]:
    """Extract matrix values from question text"""
    pattern = r'\[\[(.*?)\]\]'
    match = re.search(pattern, question)
    if not match:
        raise ValueError(f"Could not extract matrix from: {question}")

    matrix_str = match.group(1)
    rows = re.split(r'\],\s*\[', matrix_str)

    matrix = []
    for row in rows:
        row = row.strip('[]')
        values = [float(x.strip()) for x in row.split(',')]
        matrix.append(values)

    return matrix

def get_correct_answer(question: str) -> float:
    """Get the mathematically correct answer for a question"""
    matrix = extract_matrix_from_question(question)
    if len(matrix) == 2:
        return manual_determinant_2x2(matrix)
    elif len(matrix) == 3:
        return manual_determinant_3x3(matrix)
    else:
        raise ValueError(f"Unsupported matrix size: {len(matrix)}x{len(matrix[0])}")

print("✅ Manual verification functions loaded")

# =========================
# Enhanced Assessment with Debugging
# =========================
def assess_with_debugging(problems: List[Dict[str, Any]], rel_tol: float = 1e-6):
    """
    Assess problems with comprehensive debugging output
    """
    print("🚀 STARTING ENHANCED MATH SOLVER ASSESSMENT")
    print("=" * 60)

    results = []
    correct_count = 0

    for i, problem in enumerate(problems, 1):
        question = problem["question"]
        given_answer = float(problem["given_answer"])

        print(f"\n📝 PROBLEM {i}")
        print(f"Question: {question}")
        print(f"Given expected answer: {given_answer}")

        # Get mathematically correct answer
        try:
            correct_answer = get_correct_answer(question)
            print(f"✅ Manual calculation: {correct_answer}")

            # Check if given answer is correct
            if not math.isclose(given_answer, correct_answer, rel_tol=rel_tol):
                print(f"⚠️  WARNING: Given expected answer ({given_answer}) is WRONG!")
                print(f"    Correct answer should be: {correct_answer}")
        except Exception as e:
            print(f"❌ Could not verify manually: {e}")
            correct_answer = given_answer  # Fall back to given answer

        # Solve with WizardMath
        solver_result = solve_with_wizard(question, debug=True)

        if not solver_result["success"]:
            print(f"❌ Solver failed: {solver_result['error']}")
            results.append({
                "problem_num": i,
                "question": question,
                "correct_answer": correct_answer,
                "solver_answer": None,
                "is_correct": False,
                "error": solver_result["error"]
            })
            continue

        # Parse the answer
        parse_result = parse_answer_regex_only(solver_result["raw_output"], debug=True)

        if not parse_result["parsing_success"]:
            print(f"❌ Could not parse answer from solver output")
            print(f"Raw output was: {solver_result['raw_output']}")
            results.append({
                "problem_num": i,
                "question": question,
                "correct_answer": correct_answer,
                "solver_answer": None,
                "is_correct": False,
                "error": "Parsing failed"
            })
            continue

        solver_answer = parse_result["final_answer"]
        is_correct = math.isclose(solver_answer, correct_answer, rel_tol=rel_tol)

        print(f"\n📊 RESULTS:")
        print(f"Solver answer: {solver_answer}")
        print(f"Correct answer: {correct_answer}")
        print(f"Status: {'✅ CORRECT' if is_correct else '❌ INCORRECT'}")

        if is_correct:
            correct_count += 1

        results.append({
            "problem_num": i,
            "question": question,
            "correct_answer": correct_answer,
            "solver_answer": solver_answer,
            "is_correct": is_correct,
            "error": None
        })

    # Final summary
    total_problems = len(problems)
    accuracy = (correct_count / total_problems) * 100

    print(f"\n🎯 FINAL SUMMARY")
    print(f"=" * 60)
    print(f"Total problems: {total_problems}")
    print(f"Correct answers: {correct_count}")
    print(f"Accuracy: {accuracy:.1f}%")

    return results

print("✅ Assessment function loaded")

# =========================
# Test Problems (with corrected expected answers)
# =========================
sample_problems = [
    {"question": "Compute the determinant of the matrix [[8, 10], [-11, -5]]", "given_answer": 70},
    {"question": "Compute the determinant of the matrix [[-2, 1], [7, -1]]", "given_answer": -5},
    {"question": "Calculate the determinant of the matrix [[4, 6, 2], [6, -6, -5], [3, 3, -1]]", "given_answer": 102},
    {"question": "Calculate the determinant of the matrix [[0, 2, 6], [-4, -4, -5], [3, 0, 1]]", "given_answer": 50},  # CORRECTED: was 60, should be 50
    {"question": "Calculate the determinant of the matrix [[-7, -4, 6], [0, -4, -7], [-7, 1, 4]]", "given_answer": -301},
]

print("📋 Test problems loaded:")
for i, problem in enumerate(sample_problems, 1):
    print(f"{i}. {problem['question']} (Expected: {problem['given_answer']})")

# =========================
# Run the Enhanced Assessment
# =========================
print("🎯 RUNNING IMPROVED MATH SOLVER ASSESSMENT")
print("\nKey improvements over original:")
print("- ❌ Removed problematic parser LLM")
print("- 🔍 Enhanced debugging output")
print("- ✅ Corrected test case expected answers")
print("- 🎯 Better prompting strategy")
print("- 🔬 Manual verification of all answers")

results = assess_with_debugging(sample_problems)

print("\n" + "="*60)
print("ASSESSMENT COMPLETE!")
print("Check the detailed output above to see:")
print("- Exact prompts sent to the solver")
print("- Raw solver outputs (no parser interference)")
print("- Manual verification of each answer")
print("- Clear identification of any remaining issues")

### version 3-improved parser

In [ ]:
"""
FINAL LLM Math Solver - Trust the AI Expert!
Complete refactored solution in one cell - ready to copy/paste to notebook

Key Features:
- Trust WizardMath completely to solve determinant problems
- Fixed answer extraction to grab FINAL answers (not intermediate calculations)
- No manual calculation interference
- Multiple extraction patterns with proper priority
- Clean, single-cell implementation
"""

import re
import math
import torch
from typing import List, Dict, Any
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, logging as hf_logging

# Silence warnings
hf_logging.set_verbosity_error()

print(f"🚀 FINAL LLM MATH SOLVER")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# =========================
# Load WizardMath Expert
# =========================
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

print("Loading WizardMath Expert...")
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_MODEL_DIR, dtype=torch.float16, device_map="auto")
model.eval()

solver_pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False,
    pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id
)
print("✅ WizardMath Expert Ready!")

# =========================
# Expert Solver Function
# =========================
def solve_with_expert(question: str, debug: bool = True):
    prompt = (
        "You are WizardMath, a world-class expert in mathematics and linear algebra.\n"
        "Solve this determinant problem step by step with clear calculations.\n"
        "Show your work, then provide the final numerical answer.\n"
        "End with: Final Answer: [your numerical result]\n\n"
        f"Problem: {question}\n\nSolution:"
    )

    if debug:
        print(f"\n🧠 EXPERT PROMPT:\n{'='*60}\n{prompt}\n{'='*60}")

    output = solver_pipe(prompt, return_full_text=False)[0]["generated_text"].strip()

    if debug:
        print(f"\n🧠 EXPERT SOLUTION:\n{'='*60}\n{output}\n{'='*60}")

    return output

# =========================
# Smart Answer Extraction
# =========================
def extract_final_answer(text: str, debug: bool = True):
    # Priority 1: Explicit final answer statements
    final_patterns = [
        r"Final\s*Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"The\s*final\s*answer\s*is\s*:?\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"The\s*answer\s*is\s*:?\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"Answer\s*:\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
        r"Therefore,?\s*(?:the\s*)?(?:determinant\s*)?(?:is\s*)?:?\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)",
    ]

    # Priority 2: Final calculation results (last equals)
    calc_patterns = [
        r"=\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)\.?\s*$",
        r"=\s*([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)\.?\s*(?:The|Answer|Final|Therefore)",
    ]

    answer = None
    method = None

    # Try explicit final answers first
    for i, pattern in enumerate(final_patterns):
        matches = list(re.finditer(pattern, text, re.IGNORECASE))
        if matches:
            try:
                answer = float(matches[-1].group(1))  # Take last match
                method = f"Explicit Final Pattern {i+1}"
                break
            except (ValueError, TypeError):
                continue

    # Fallback to calculation results
    if answer is None:
        for i, pattern in enumerate(calc_patterns):
            matches = list(re.finditer(pattern, text, re.IGNORECASE | re.MULTILINE))
            if matches:
                try:
                    answer = float(matches[-1].group(1))  # Take last match
                    method = f"Calculation Pattern {i+1}"
                    break
                except (ValueError, TypeError):
                    continue

    # Last resort: final standalone number
    if answer is None:
        numbers = re.findall(r"(?:^|\s)([+-]?(?:\d+(?:\.\d+)?|\.\d+)(?:[eE][+-]?\d+)?)(?:\.|$|\s)", text, re.MULTILINE)
        if numbers:
            try:
                answer = float(numbers[-1])
                method = "Last Standalone Number"
            except (ValueError, TypeError):
                pass

    if debug:
        print(f"\n🔍 EXTRACTION: Answer={answer}, Method={method}")

    return answer, method

# =========================
# Assessment Function
# =========================
def assess_expert(problems: List[Dict[str, Any]], debug: bool = True):
    print(f"\n🧠 EXPERT ASSESSMENT - Trusting WizardMath!")
    print("=" * 60)

    results = []
    correct = 0

    for i, prob in enumerate(problems, 1):
        question = prob["question"]
        expected = float(prob["given_answer"])

        print(f"\n📝 PROBLEM {i}: {question}")
        print(f"Expected: {expected}")

        # Let expert solve
        solution = solve_with_expert(question, debug)
        expert_answer, extraction_method = extract_final_answer(solution, debug)

        if expert_answer is None:
            print(f"❌ Could not extract answer from solution")
            results.append({"problem": i, "expected": expected, "expert": None, "correct": False, "method": "failed"})
            continue

        is_correct = math.isclose(expert_answer, expected, rel_tol=1e-6, abs_tol=1e-6)

        print(f"📊 Expert: {expert_answer} | Expected: {expected} | {'✅ MATCH' if is_correct else '❌ DIFFERENT'}")
        print(f"Extraction: {extraction_method}")

        if not is_correct:
            print(f"💡 Expert calculated {expert_answer} vs expected {expected} - Expert might be right!")

        if is_correct:
            correct += 1

        results.append({
            "problem": i, "expected": expected, "expert": expert_answer,
            "correct": is_correct, "method": extraction_method
        })

    accuracy = (correct / len(problems)) * 100
    print(f"\n🎯 SUMMARY: {correct}/{len(problems)} matches ({accuracy:.1f}%)")
    print("Remember: If accuracy is low, the expert might be right and expected answers wrong!")

    return results

# =========================
# Test Problems & Execution
# =========================
test_problems = [
    {"question": "Compute the determinant of the matrix [[8, 10], [-11, -5]]", "given_answer": 70},
    {"question": "Compute the determinant of the matrix [[-2, 1], [7, -1]]", "given_answer": -5},
    {"question": "Calculate the determinant of the matrix [[4, 6, 2], [6, -6, -5], [3, 3, -1]]", "given_answer": 102},
    {"question": "Calculate the determinant of the matrix [[0, 2, 6], [-4, -4, -5], [3, 0, 1]]", "given_answer": 50},
    {"question": "Calculate the determinant of the matrix [[-7, -4, 6], [0, -4, -7], [-7, 1, 4]]", "given_answer": -301},
]

print("\n🎯 RUNNING FINAL ASSESSMENT")
print("Key improvements:")
print("- ✅ Trust WizardMath expert completely")
print("- ✅ Fixed answer extraction (prioritize final conclusions)")
print("- ✅ No manual calculation interference")
print("- ✅ Smart pattern matching with fallbacks")

# Run the assessment
results = assess_expert(test_problems, debug=True)

print(f"\n{'='*60}")
print("🎉 ASSESSMENT COMPLETE!")
print("The math expert has provided detailed solutions.")
print("Trust the expert's step-by-step reasoning!")
print(f"{'='*60}")

🚀 FINAL LLM MATH SOLVER
CUDA available: True
Using GPU: Tesla T4
Loading WizardMath Expert...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ WizardMath Expert Ready!

🎯 RUNNING FINAL ASSESSMENT
Key improvements:
- ✅ Trust WizardMath expert completely
- ✅ Fixed answer extraction (prioritize final conclusions)
- ✅ No manual calculation interference
- ✅ Smart pattern matching with fallbacks

🧠 EXPERT ASSESSMENT - Trusting WizardMath!

📝 PROBLEM 1: Compute the determinant of the matrix [[8, 10], [-11, -5]]
Expected: 70.0

🧠 EXPERT PROMPT:
You are WizardMath, a world-class expert in mathematics and linear algebra.
Solve this determinant problem step by step with clear calculations.
Show your work, then provide the final numerical answer.
End with: Final Answer: [your numerical result]

Problem: Compute the determinant of the matrix [[8, 10], [-11, -5]]

Solution:

🧠 EXPERT SOLUTION:
= (8)*(-5) - (10)*(-11) = -40 - -110 = 70.

The answer is: 70.

🔍 EXTRACTION: Answer=70.0, Method=Explicit Final Pattern 3
📊 Expert: 70.0 | Expected: 70.0 | ✅ MATCH
Extraction: Explicit Final Pattern 3

📝 PROBLEM 2: Compute the determinant of the m

## Final version

###Connection to firebase

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

# If using Drive:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/drive/MyDrive/smart-assessor-fdeaa-firebase-adminsdk-fbsvc-cd76bf92b6.json"

os.environ["FIREBASE_PROJECT_ID"] = "smart-assessor-fdeaa"



Mounted at /content/drive


In [2]:
!pip -q install transformers peft accelerate bitsandbytes langchain langchain_community google-cloud-firestore google-auth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [4]:
from google.cloud import firestore
db = firestore.Client(project=os.getenv("FIREBASE_PROJECT_ID"))
db.collection("hello").document("colab").set({"ok": True})
print("Connected to Firestore.")


Connected to Firestore.


####Load the adapted model once

In [5]:
# --- KEEP: core imports + GPU check ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# --- KEEP: model path and load ---
MERGED_MODEL_DIR = "/content/drive/MyDrive/models/WizardMath-7B-V1.1-SmartAssessor-Merged"

print("Loading WizardMath tokenizer + model from Google Drive...")
tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("WizardMath loaded ✅")

# --- KEEP: build a deterministic generation pipeline ---
solver_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

# --- NEW: system prompt and a .chat() wrapper (to match our worker loop) ---
SOLVER_SYS = (
    "You are WizardMath, an expert in linear algebra determinants. "
    "Solve the problem. "
    "Keep the reasoning to one short paragraph. "
    "Then compare with the provided student_answer. "
    "If it matches your result, write: ✅ Correct. "
    "If it does not, write: ❌ Incorrect — Correct answer: <number>, plus one short, brief calculation of the correct answer."
)

class ColabLLMWorker:
    def chat(self, text: str) -> str:
        # You can customize this prompt if your tasks are not math-only (see note below)
        prompt = f"{SOLVER_SYS}\n\nProblem: {text}\n\nSolution:"
        out = solver_pipe(prompt, return_full_text=False)[0]["generated_text"].strip()
        return out

_llm = ColabLLMWorker()
print("LLM is warm and ready. You can start the worker loop now.")


CUDA available: True
Using GPU: Tesla T4
Loading WizardMath tokenizer + model from Google Drive...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


WizardMath loaded ✅
LLM is warm and ready. You can start the worker loop now.


####Loop through tasks sent by the backend

In [7]:
from google.cloud import firestore
from datetime import datetime
import json
import math
import os
import re
import sys
import time
import traceback
from typing import Any, Dict, List

PROJECT_ID = os.getenv("FIREBASE_PROJECT_ID")
db = firestore.Client(project=PROJECT_ID)

DEBUG = True
MAX_PREVIEW = 500

STUDENT_PREFIXES = ("student result", "student answer", "student response")

ANSWER_PATTERNS = (
    r"Final\s+Answer\s*[:=\-]\s*([-+]?\d*\.?\d+)",
    r"The\s+answer\s+is\s*[:=\-]\s*([-+]?\d*\.?\d+)",
    r"Answer\s*[:=\-]\s*([-+]?\d*\.?\d+)",
)


def _ts() -> str:
    return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S.%fZ")


def _preview(obj: Any, max_len: int = MAX_PREVIEW) -> str:
    try:
        s = obj if isinstance(obj, str) else json.dumps(obj, ensure_ascii=False, default=str)
    except Exception:
        s = str(obj)
    if len(s) > max_len:
        return s[:max_len] + f"... [truncated {len(s)-max_len} chars]"
    return s


def dbg(*args: Any) -> None:
    if DEBUG:
        print(f"[{_ts()}][DEBUG]", *args, file=sys.stdout, flush=True)


def _compare(pred, student, rel_tol: float = 1e-6):
    dbg("_compare() called with:", f"pred={_preview(pred)}", f"student={_preview(student)}", f"rel_tol={rel_tol}")
    try:
        a = float(str(pred).strip())
        b = float(str(student).strip())
        is_ok = math.isclose(a, b, rel_tol=rel_tol)
        delta = a - b
        rel_err = abs(delta) / max(1.0, abs(b))
        dbg("_compare() numeric result:", f"is_ok={is_ok}", f"delta={delta}", f"rel_err={rel_err}")
        return is_ok, delta, rel_err
    except Exception as ex:
        dbg("_compare() fell back to string compare due to:", repr(ex))
        s_pred = str(pred).strip()
        s_stu = str(student).strip()
        res = (s_pred == s_stu), None, None
        dbg("_compare() string result:", f"is_ok={res[0]}")
        return res


def _split_json_first_line(text: str):
    # Retained for potential legacy callers; now used only for logging.
    dbg("_split_json_first_line() called. text preview:", _preview(text))
    lines = text.splitlines()
    if not lines:
        dbg("_split_json_first_line(): no lines -> (None, '')")
        return None, ""
    try:
        j = json.loads(lines[0])
        rest = "\n".join(lines[1:]).strip()
        dbg("_split_json_first_line(): parsed JSON on first line:", _preview(j), "rest preview:", _preview(rest))
        return j, rest
    except Exception as ex:
        dbg("_split_json_first_line(): JSON parse failed on first line:", repr(ex), "-> returning (None, full_text)")
        return None, text


def _extract_tasks(piece: str) -> List[Dict[str, str]]:
    dbg("_extract_tasks() piece preview:", _preview(piece))
    lines = [ln.strip() for ln in piece.splitlines() if ln.strip()]
    tasks: List[Dict[str, str]] = []
    pending: List[str] = []

    for line in lines:
        lower = line.lower()
        if any(lower.startswith(prefix) for prefix in STUDENT_PREFIXES) and ":" in line:
            label, value = line.split(":", 1)
            problem_text = "\n".join(pending).strip()
            raw_block = "\n".join(pending + [line]).strip()
            tasks.append(
                {
                    "problem": problem_text or piece.strip(),
                    "student_label": label.strip(),
                    "student_answer": value.strip(),
                    "raw_input": raw_block,
                }
            )
            dbg("_extract_tasks() captured task:", _preview(tasks[-1]))
            pending = []
        else:
            pending.append(line)

    if not tasks and piece.strip():
        # Fall back to treating the whole chunk as one task with no student answer.
        tasks.append(
            {
                "problem": piece.strip(),
                "student_label": "",
                "student_answer": "",
                "raw_input": piece.strip(),
            }
        )
        dbg("_extract_tasks() fallback single-task:", _preview(tasks[-1]))

    return tasks


def _parse_model_answer(raw: str):
    dbg("_parse_model_answer() raw preview:", _preview(raw))
    predicted = None
    explanation = raw.strip()

    for pattern in ANSWER_PATTERNS:
        match = re.search(pattern, raw, flags=re.IGNORECASE)
        if match:
            predicted = match.group(1).strip()
            explanation = raw[: match.start()].strip()
            dbg("_parse_model_answer() matched pattern:", pattern, "predicted:", predicted)
            break

    if not predicted:
        numbers = re.findall(r"[-+]?\d*\.?\d+", raw)
        if numbers:
            predicted = numbers[-1]
            dbg("_parse_model_answer() fallback last number:", predicted)

    return predicted, explanation


def _format_result(is_ok: bool, delta, rel_err, predicted, student_answer, explanation: str):
    if is_ok:
        body = explanation or "Excellent work—your reasoning matches the expected solution."
        return (
            "✅ Correct\n"
            f"Student answer: {student_answer}\n"
        )

    lines = [
        "❌ Incorrect",
        f"Correct answer: {predicted}",
        f"Student answer: {student_answer}",
    ]
    lines.append(f"Calculation: {explanation or 'Review the corrected calculation above.'}")
    return "\n".join(lines)


def _process_job(doc_ref, job):
    dbg("_process_job() start. job keys:", list((job or {}).keys()))
    try:
        req = (job or {}).get("request", {})
        dbg("_process_job() request preview:", _preview(req))

        if req.get("type") != "grade_pdf":
            dbg("_process_job(): unknown request type:", req.get("type"))
            doc_ref.update(
                {
                    "status": "error",
                    "updatedAt": firestore.SERVER_TIMESTAMP,
                    "error": f"Unknown request type: {req.get('type')}",
                    "result": {"text": ""},
                }
            )
            return

        payload = req.get("payload", {}) or {}
        chunks = payload.get("chunks")
        student_text = payload.get("student_text")
        dbg(
            "_process_job() payload fields:",
            f"chunks_len={len(chunks) if chunks else 0}",
            f"student_text_present={student_text is not None}",
        )

        tasks: List[Dict[str, str]] = []
        if chunks:
            for chunk in chunks:
                per_chunk = _extract_tasks(chunk)
                tasks.extend(per_chunk)
        elif student_text:
            tasks.extend(_extract_tasks(student_text))

        if not tasks:
            dbg("_process_job() no tasks extracted; adding empty placeholder.")
            tasks = [
                {
                    "problem": "",
                    "student_label": "",
                    "student_answer": "",
                    "raw_input": student_text or "",
                }
            ]

        outputs = []
        total = len(tasks)
        for i, task in enumerate(tasks, 1):
            raw_input = task["raw_input"]
            dbg(f"_process_job() -> calling _llm.chat for task {i}/{total}. input preview:", _preview(raw_input))
            print(f"[Worker] task {i}/{total}")
            try:
                raw_response = _llm.chat(raw_input)
            except Exception as chat_ex:
                dbg(f"_process_job() _llm.chat raised for task {i}:", repr(chat_ex))
                raw_response = f"[LLM ERROR] {chat_ex}"

            dbg(f"_process_job() <- _llm.chat result for task {i}:", _preview(raw_response))

            predicted, explanation = _parse_model_answer(raw_response)
            student_answer = task.get("student_answer", "")

            if predicted and student_answer:
                is_ok, delta, rel_err = _compare(predicted, student_answer)
                dbg(
                    f"_process_job() compare() result for task {i}:",
                    f"is_ok={is_ok}",
                    f"delta={delta}",
                    f"rel_err={rel_err}",
                )
                piece_out = _format_result(is_ok, delta, rel_err, predicted, student_answer, explanation)
            elif student_answer:
                dbg(f"_process_job() could not extract model answer for task {i}.")
                piece_out = (
                    "⚠️ Could not extract model answer from LLM output.\n"
                    f"Student answer: {student_answer}\n"
                    f"Model response:\n{raw_response}"
                )
            else:
                dbg(f"_process_job() no student answer for task {i}; returning model response only.")
                piece_out = raw_response

            dbg(f"_process_job() formatted piece_out for task {i}:", _preview(piece_out))
            outputs.append(piece_out)

        combined = "\n\n".join(outputs)
        result_obj = {"text": combined, "meta": {"worker": "colab", "ts": datetime.utcnow().isoformat()}}
        dbg("_process_job() final combined output preview:", _preview(combined))
        dbg("_process_job() updating Firestore to 'done'.")
        doc_ref.update(
            {
                "status": "done",
                "updatedAt": firestore.SERVER_TIMESTAMP,
                "result": result_obj,
            }
        )

    except Exception as e:
        print("[Worker] ERROR:", e)
        traceback.print_exc()
        dbg("_process_job() exception caught; updating Firestore to 'error':", repr(e))
        doc_ref.update(
            {
                "status": "error",
                "updatedAt": firestore.SERVER_TIMESTAMP,
                "error": str(e),
                "result": {"text": ""},
            }
        )


def poll_loop(sleep_if_empty: float = 1.0):
    print(f"[Worker] Project={PROJECT_ID} polling...")
    dbg("poll_loop() started with sleep_if_empty:", sleep_if_empty)
    while True:
        any_found = False
        try:
            dbg("poll_loop() querying jobs: status=='queued', limit=3")
            query = db.collection("jobs").where("status", "==", "queued").order_by("createdAt").limit(3)
            queued_docs = list(query.stream())
            dbg(f"poll_loop() fetched {len(queued_docs)} queued job(s).")
        except Exception as q_ex:
            dbg("poll_loop() Firestore query failed:", repr(q_ex))
            time.sleep(sleep_if_empty)
            continue

        for snap in queued_docs:
            any_found = True
            doc_ref = snap.reference
            job_dict = snap.to_dict() or {}
            dbg("poll_loop() picked job id:", doc_ref.id, "job preview:", _preview(job_dict))
            try:
                dbg("poll_loop() setting job status -> 'running'")
                doc_ref.update({"status": "running", "updatedAt": firestore.SERVER_TIMESTAMP})
            except Exception as up_ex:
                dbg("poll_loop() failed to set 'running' (skipping):", repr(up_ex))
                continue

            _process_job(doc_ref, job_dict)

        if not any_found:
            dbg("poll_loop() no jobs found; sleeping", sleep_if_empty, "sec")
            time.sleep(sleep_if_empty)


poll_loop()


[Worker] Project=smart-assessor-fdeaa polling...
[2025-09-18 08:29:02.059963Z][DEBUG] poll_loop() started with sleep_if_empty: 1.0
[2025-09-18 08:29:02.060562Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3


/tmp/ipython-input-935820407.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S.%fZ")


[2025-09-18 08:29:03.931680Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:29:03.932386Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:29:04.933064Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:29:05.277174Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:29:05.277860Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:29:06.278549Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:29:06.690812Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:29:06.691480Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:29:07.692194Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:29:08.045935Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:29:08.046604Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:29:09.047555Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:29:09

/tmp/ipython-input-935820407.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  result_obj = {"text": combined, "meta": {"worker": "colab", "ts": datetime.utcnow().isoformat()}}


[2025-09-18 08:30:54.719682Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:30:55.035647Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:30:55.036287Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:30:56.036916Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:30:56.347230Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:30:56.347849Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:30:57.348818Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:30:57.661735Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:30:57.662407Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:30:58.663401Z][DEBUG] poll_loop() querying jobs: status=='queued', limit=3
[2025-09-18 08:30:58.976456Z][DEBUG] poll_loop() fetched 0 queued job(s).
[2025-09-18 08:30:58.977120Z][DEBUG] poll_loop() no jobs found; sleeping 1.0 sec
[2025-09-18 08:30:59

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[Worker] task 1/3
[2025-09-18 08:33:41.154364Z][DEBUG] _process_job() <- _llm.chat result for task 1: To compute the determinant of a 3x3 matrix [[a, b, c], [d, e, f], [g, h, i]], we use the formula: det = a*(e*i - f*h) - b*(d*i - f*g) + c*(d*h - e*g).
In this case, a = 2, b = 1, c = 3, d = 0, e = 4, f = -1, g = 1, h = 2, i = 5.
So, det = 2*(4*5 - -1*2) - 1*(0*5 - -1*1) + 3*(0*2 - 4*1) = 2*(22) - 1*(1) + 3*(-4) = 44 - 1 + -12 = 31.

The answer is: 31.
[2025-09-18 08:33:41.155174Z][DEBUG] _parse_model_answer() raw preview: To compute the determinant of a 3x3 matrix [[a, b, c], [d, e, f], [g, h, i]], we use the formula: det = a*(e*i - f*h) - b*(d*i - f*g) + c*(d*h - e*g).
In this case, a = 2, b = 1, c = 3, d = 0, e = 4, f = -1, g = 1, h = 2, i = 5.
So, det = 2*(4*5 - -1*2) - 1*(0*5 - -1*1) + 3*(0*2 - 4*1) = 2*(22) - 1*(1) + 3*(-4) = 44 - 1 + -12 = 31.

The answer is: 31.
[2025-09-18 08:33:41.156156Z][DEBUG] _parse_model_answer() matched pattern: The\s+answer\s+is\s*[:=\-]\s*([-+]?\d*\.?\

KeyboardInterrupt: 